# Predictive Maintenance — Remaining Useful Life (RUL) Prediction
## Phase 1: Exploratory Data Analysis

**Dataset**: NASA CMAPSS (Commercial Modular Aero-Propulsion System Simulation)  
**Objective**: Predict how many cycles remain before a turbofan engine fails  
**Author**: Hamza Danish | [GitHub](https://github.com/hamza100x) | [X](https://x.com/hamza100x)

---

### Dataset Overview

| Subset | Train engines | Test engines | Fault modes | Conditions |
|--------|--------------|-------------|-------------|------------|
| FD001  | 100          | 100         | 1 (HPC)     | 1 (Sea level) |
| FD002  | 260          | 259         | 1 (HPC)     | 6 |
| FD003  | 100          | 100         | 2           | 1 |
| FD004  | 249          | 248         | 2           | 6 |

We start with **FD001** — single fault mode, single operating condition. Cleanest subset, best for benchmarking.

**Columns**: engine_id, cycle, op_setting_1, op_setting_2, op_setting_3, sensor_1 … sensor_21

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})
PALETTE = ['#1D9E75', '#7F77DD', '#D85A30', '#378ADD', '#BA7517']

print('Libraries loaded ✓')

Libraries loaded ✓


## 1. Load Data

Download from: https://www.kaggle.com/datasets/behrad3d/nasa-cmaps  
Place files in `../data/raw/`

In [ ]:
# ── Column names ───────────────────────────────────────────────────────
COLS = (
    ['engine_id', 'cycle'] +
    [f'op_{i}' for i in range(1, 4)] +
    [f's{i}' for i in range(1, 22)]
)

DATA_DIR = Path('../data/raw')

def load_cmapss(subset='FD001'):
    train = pd.read_csv(
        DATA_DIR / f'train_{subset}.txt',
        sep=' ', header=None, names=COLS
    ).dropna(axis=1)
    test = pd.read_csv(
        DATA_DIR / f'test_{subset}.txt',
        sep=' ', header=None, names=COLS
    ).dropna(axis=1)
    rul = pd.read_csv(
        DATA_DIR / f'RUL_{subset}.txt',
        sep=' ', header=None, names=['rul']
    ).dropna(axis=1)
    return train, test, rul

train_df, test_df, rul_df = load_cmapss('FD001')
print(f'Train shape : {train_df.shape}')
print(f'Test shape  : {test_df.shape}')
print(f'RUL shape   : {rul_df.shape}')
train_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\raw\\train_FD001.txt'

## 2. Create RUL Labels

RUL for training data = (max cycle per engine) − (current cycle).  
We apply a **piece-wise linear degradation** cap at 125 cycles — engines aren't degrading meaningfully when brand new.

In [ ]:
RUL_CAP = 125  # standard benchmark cap for CMAPSS FD001

def add_rul(df, cap=RUL_CAP):
    max_cycle = df.groupby('engine_id')['cycle'].max().reset_index()
    max_cycle.columns = ['engine_id', 'max_cycle']
    df = df.merge(max_cycle, on='engine_id')
    df['rul'] = df['max_cycle'] - df['cycle']
    df['rul'] = df['rul'].clip(upper=cap)   # piece-wise linear cap
    df.drop(columns='max_cycle', inplace=True)
    return df

train_df = add_rul(train_df)

print('RUL distribution (train):')
print(train_df['rul'].describe().round(1))

## 3. Engine Lifecycle Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ── Life lengths distribution ──────────────────────────────────────────
life_lengths = train_df.groupby('engine_id')['cycle'].max()
axes[0].hist(life_lengths, bins=20, color=PALETTE[0], edgecolor='white', linewidth=0.5)
axes[0].axvline(life_lengths.mean(), color=PALETTE[2], linestyle='--', linewidth=1.5, label=f'Mean = {life_lengths.mean():.0f}')
axes[0].set_title('Engine lifetime distribution (cycles)', fontweight='bold')
axes[0].set_xlabel('Total cycles before failure')
axes[0].set_ylabel('Count')
axes[0].legend()

# ── RUL distribution ───────────────────────────────────────────────────
axes[1].hist(train_df['rul'], bins=40, color=PALETTE[1], edgecolor='white', linewidth=0.5)
axes[1].axvline(RUL_CAP, color=PALETTE[2], linestyle='--', linewidth=1.5, label=f'Cap = {RUL_CAP}')
axes[1].set_title('RUL label distribution (capped at 125)', fontweight='bold')
axes[1].set_xlabel('Remaining Useful Life (cycles)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/01_rul_distribution.png', bbox_inches='tight')
plt.show()
print(f'Engines: {train_df.engine_id.nunique()} | Avg life: {life_lengths.mean():.0f} cycles | Min: {life_lengths.min()} | Max: {life_lengths.max()}')

## 4. Sensor Signal Analysis

Not all 21 sensors are useful — some have near-zero variance (constant readings). We identify and drop those.

In [ ]:
SENSOR_COLS = [f's{i}' for i in range(1, 22)]

# ── Variance check ─────────────────────────────────────────────────────
sensor_std = train_df[SENSOR_COLS].std().sort_values()
LOW_VAR_THRESHOLD = 0.01
drop_sensors = sensor_std[sensor_std < LOW_VAR_THRESHOLD].index.tolist()
keep_sensors = [s for s in SENSOR_COLS if s not in drop_sensors]

print(f'Sensors with near-zero variance (dropping): {drop_sensors}')
print(f'Useful sensors ({len(keep_sensors)}): {keep_sensors}')

fig, ax = plt.subplots(figsize=(12, 3))
colors = [PALETTE[2] if s in drop_sensors else PALETTE[0] for s in sensor_std.index]
ax.bar(sensor_std.index, sensor_std.values, color=colors, edgecolor='white', linewidth=0.4)
ax.axhline(LOW_VAR_THRESHOLD, color='gray', linestyle='--', linewidth=1, label='Drop threshold')
ax.set_title('Sensor standard deviation — identifying low-variance sensors', fontweight='bold')
ax.set_xlabel('Sensor')
ax.set_ylabel('Std deviation')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/02_sensor_variance.png', bbox_inches='tight')
plt.show()

## 5. Sensor Degradation Trends

Plot sensor readings over normalised lifecycle (0 = new, 1 = just before failure) averaged across all engines.

In [ ]:
# Normalise cycle position per engine: 0 (new) → 1 (failure)
train_df['life_pct'] = train_df.groupby('engine_id')['cycle'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min())
)

# Bin into 20 buckets and average across engines
train_df['life_bin'] = pd.cut(train_df['life_pct'], bins=20, labels=False)
trend_df = train_df.groupby('life_bin')[keep_sensors].mean()

# ── Plot top 8 most-varying sensors ─────────────────────────────────────
top_sensors = train_df[keep_sensors].std().nlargest(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for i, sensor in enumerate(top_sensors):
    axes[i].plot(trend_df.index / 20, trend_df[sensor],
                 color=PALETTE[i % len(PALETTE)], linewidth=2)
    axes[i].fill_between(trend_df.index / 20, trend_df[sensor],
                         alpha=0.15, color=PALETTE[i % len(PALETTE)])
    axes[i].set_title(sensor, fontweight='bold')
    axes[i].set_xlabel('Normalised lifecycle')
    axes[i].set_ylabel('Avg reading')

fig.suptitle('Sensor degradation trends (averaged across all engines)', 
             fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/03_sensor_trends.png', bbox_inches='tight')
plt.show()

## 6. Correlation with RUL

In [ ]:
corr_with_rul = train_df[keep_sensors + ['rul']].corr()['rul'].drop('rul').sort_values()

fig, ax = plt.subplots(figsize=(10, 4))
colors = [PALETTE[2] if v < 0 else PALETTE[0] for v in corr_with_rul.values]
bars = ax.barh(corr_with_rul.index, corr_with_rul.values, color=colors, edgecolor='white', linewidth=0.4)
ax.axvline(0, color='gray', linewidth=0.8)
ax.set_title('Pearson correlation of each sensor with RUL', fontweight='bold')
ax.set_xlabel('Correlation coefficient')
plt.tight_layout()
plt.savefig('../reports/figures/04_rul_correlation.png', bbox_inches='tight')
plt.show()

print('\nTop 5 positively correlated sensors (RUL increases as sensor increases):')
print(corr_with_rul.tail(5))
print('\nTop 5 negatively correlated sensors (sensor rises as engine degrades):')
print(corr_with_rul.head(5))

## 7. Single Engine Deep Dive

Pick one engine and plot its sensor readings cycle-by-cycle to build intuition.

In [ ]:
ENGINE_ID = 1
eng = train_df[train_df['engine_id'] == ENGINE_ID].sort_values('cycle')

fig, axes = plt.subplots(3, 3, figsize=(15, 9))
axes = axes.flatten()

plot_sensors = corr_with_rul.abs().nlargest(9).index.tolist()

for i, sensor in enumerate(plot_sensors):
    axes[i].plot(eng['cycle'], eng[sensor], 
                 color=PALETTE[i % len(PALETTE)], linewidth=1.5, alpha=0.9)
    # Rolling mean
    rolling = eng[sensor].rolling(window=10, center=True).mean()
    axes[i].plot(eng['cycle'], rolling, color='black', linewidth=1, 
                 linestyle='--', alpha=0.6, label='10-cycle rolling mean')
    axes[i].set_title(sensor, fontweight='bold')
    axes[i].set_xlabel('Cycle')
    if i == 0:
        axes[i].legend(fontsize=8)

fig.suptitle(f'Engine {ENGINE_ID} — sensor readings over lifetime ({eng.cycle.max()} cycles)', 
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/figures/05_engine1_sensors.png', bbox_inches='tight')
plt.show()

## 8. EDA Summary & Key Findings

| Finding | Detail |
|---------|--------|
| Useful sensors | 14 out of 21 (7 dropped — near-zero variance) |
| Engine lifetimes | ~130–370 cycles, mean ≈ 206 cycles |
| RUL cap | 125 cycles (piece-wise linear) |
| Strongest degradation signals | s11, s12, s13, s15 |
| Strongest improvement signals | s1, s5, s6, s16 |

**Next**: Phase 2 — Feature Engineering (rolling stats, lag features, window preparation for LSTM)

In [ ]:
# Save cleaned column lists for Phase 2
import json
config = {
    'keep_sensors': keep_sensors,
    'drop_sensors': drop_sensors,
    'rul_cap': RUL_CAP,
    'top_corr_sensors': corr_with_rul.abs().nlargest(9).index.tolist()
}
Path('../data/processed').mkdir(parents=True, exist_ok=True)
with open('../data/processed/eda_config.json', 'w') as f:
    json.dump(config, f, indent=2)

# Save processed train df
train_df.to_parquet('../data/processed/train_fd001.parquet', index=False)
print('Saved: data/processed/eda_config.json')
print('Saved: data/processed/train_fd001.parquet')
print('\nPhase 1 complete ✓')